## MMTHE01 - Masters Thesis

### A. Thesis - Data Pre-processing
* Step 1: Preliminary check for missing data
* Step 2: Addressing multicollinearity among features
* Step 3: Encoding Categorical Variables

#### Importing the libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os

from sklearn.preprocessing import LabelEncoder

In [2]:
# Check your current working directory
cwd = os.getcwd()

# Define your relative path
relative_path = r"Analysis"  # adjust this relative to cwd

# Build the full path
full_path = os.path.join(cwd, relative_path)

# Check if it exists before changing
if os.path.exists(full_path):
    os.chdir(full_path)
    print("Changed directory to:", full_path)
else:
    print("Folder does not exist:", full_path)

Changed directory to: C:\Users\eaber\Documents\Masters Thesis Final\Analysis


#### Importing the dataset

In [3]:
dataset = pd.read_csv('loan.csv', low_memory=False)

In [4]:
dataset.shape

(2260668, 145)

In [5]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Columns: 145 entries, id to settlement_term
dtypes: float64(105), int64(4), object(36)
memory usage: 2.4+ GB


In [6]:
dataset.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
dataset = dataset.reset_index(drop=False).rename(columns={'index': 'tmp_id'})

In [8]:
dataset.head(100).to_csv('intermediate_output/check_lending_club.csv',index=False)

### Step 1: Creating a stratified sample of the dataset

In [9]:
# Obtain the year issued
dataset['year_issued'] = dataset['issue_d'].apply(lambda x: x.split('-')[1])

In [10]:
# Map target "default_status" variable (example: default vs non-default)
dataset['default_status'] = dataset['loan_status'].map(lambda s: 1 if s in ('Charged Off', 'Default') else 0)

In [11]:
dataset['default_status'].value_counts(normalize=True) * 100

default_status
0    88.424395
1    11.575605
Name: proportion, dtype: float64

In [12]:
# Drop rows with missing key info i.e. the strata (keeps strata valid)
dataset = dataset.dropna(subset=['year_issued', 'default_status'])

In [13]:
# Define strata: combine year and target
dataset['stratum'] = dataset['year_issued'].astype(str) + '_' + dataset['default_status'].astype(str)

In [14]:
# Determine sampling fraction (example: 10% of population)
sample_frac = 0.10

In [15]:
# Stratified sampling by strata
final_dataset = (
    dataset.groupby('stratum', group_keys=False)
           .apply(lambda x: x.sample(frac=sample_frac, random_state=1), include_groups=False)
           .reset_index(drop=True)
)

In [16]:
final_dataset.head(2)

,tmp_id,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,year_issued,default_status
0,2141902,NaN,NaN,500,500,375.0,36 months,8.38,15.76,A,...,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,2007,0
1,2141992,NaN,NaN,6000,6000,1200.0,36 months,13.12,202.51,D,...,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,2007,0


In [17]:
# Check the resulting sample size
print("Sampled dataset size:", len(final_dataset))

Sampled dataset size: 226067


In [18]:
final_dataset['year_issued'].value_counts(normalize=True) * 100

year_issued
2018    21.906780
2017    19.621617
2016    19.215985
2015    18.627221
2014    10.423016
2013     5.963276
2012     2.360362
2011     0.960777
2010     0.554703
2009     0.233559
2008     0.106163
2007     0.026541
Name: proportion, dtype: float64

In [19]:
final_dataset['default_status'].value_counts(normalize=True) * 100

default_status
0    88.424228
1    11.575772
Name: proportion, dtype: float64

In [20]:
# Get remaining (unsampled) data
remaining_dataset = dataset.loc[~dataset['tmp_id'].isin(final_dataset['tmp_id'])]

In [21]:
# Keep ONLY the positive class from remaining data
remaining_positive = remaining_dataset.loc[remaining_dataset['default_status'] == 1].reset_index(drop=True)

In [22]:
remaining_positive.shape

(235517, 149)

In [23]:
remaining_positive.head(2)

,tmp_id,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,year_issued,default_status,stratum
0,5040,NaN,NaN,8000,8000,8000.0,36 months,6.46,245.05,A,...,N,NaN,NaN,NaN,NaN,NaN,NaN,2018,1,2018_1
1,25963,NaN,NaN,6000,6000,6000.0,36 months,14.47,206.44,C,...,N,NaN,NaN,NaN,NaN,NaN,NaN,2018,1,2018_1


In [24]:
remaining_positive.to_csv('remaining_positive_lending_club.csv',index=False)

### Export Data

In [25]:
# Drop tmp_id (only after all operations)
final_dataset = final_dataset.drop(columns=['tmp_id'])
remaining_positive = remaining_positive.drop(columns=['tmp_id'])

In [26]:
final_dataset.head(1)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,year_issued,default_status
0,NaN,NaN,500,500,375.0,36 months,8.38,15.76,A,A5,...,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,2007,0


In [27]:
final_dataset.to_csv('final_lending_club.csv',index=False)

In [28]:
remaining_positive.to_csv('remaining_positive_lending_club.csv',index=False)